# run_repeatability: same pose 100 times, spread in millimeters

Run notebook (needs the arm). Milestone: *Baseline repeatability measured*. Command the arm
from a varying start to the same target pose 100 times and measure where the gripper tip
ends up. Two measurements: (1) joint encoders -> FK (what the arm *thinks*), (2) an external
reference (camera on an ArUco marker in the gripper, or a dial indicator) for the real spread.
Repeat after the rigid-mounting milestone and for each control path (Python / C++ / STM32).

**TODO**
- [ ] choose the target (e.g. hover over e4) and a set of start poses
- [ ] external measurement method (ArUco on the gripper seen by the board camera?)
- [ ] run 100 trials, save CSV under data/arm/repeatability_<date>.csv
- [ ] report: mean, std, max radial spread (mm), plot below

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%run ./pick_place.ipynb

N_TRIALS = 100
TARGET_Q, _ = ik_top_down(hover_point("e4"))
OUT_CSV = REPO_ROOT / "data" / "arm" / "repeatability.csv"

In [ ]:
def run_trials(backend, read_joints, measure_external=None, n=N_TRIALS, rng=np.random.default_rng(0)):
    """TODO: for i in range(n): move to a random nearby start, then to TARGET_Q (through
    plan/densify + backend.execute), wait to settle (~1 s), record read_joints() and
    measure_external() (or NaN). Returns a DataFrame with columns trial, q0..q4, x, y, z
    (FK of read joints), ext_x, ext_y, ext_z."""
    raise NotImplementedError

def spread_mm(df, cols=("x", "y", "z")):
    """Radial spread around the mean in mm: dict(mean, std, max)."""
    P = df[list(cols)].to_numpy() * 1000
    d = np.linalg.norm(P - P.mean(axis=0), axis=1)
    return dict(mean=d.mean(), std=d.std(), max=d.max())

def plot_spread(df, cols=("x", "y")):
    P = (df[list(cols)] - df[list(cols)].mean()).to_numpy() * 1000
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(P[:, 0], P[:, 1], s=8)
    ax.set_xlabel(f"{cols[0]} (mm)"); ax.set_ylabel(f"{cols[1]} (mm)")
    ax.set_aspect("equal"); ax.set_title(f"Repeatability, n={len(P)}")
    return fig

# df = run_trials(ServoBackend(...), read_joints=...)
# df.to_csv(OUT_CSV, index=False); spread_mm(df), plot_spread(df)